In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df=pd.read_excel("uber.xlsx")

In [3]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2025-09-22,06:39:39,T1984811,Cancelled by Customer,C6701334,Auto,Dilshad Garden,Qutub Minar,1.0,Change of plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-14,11:41:45,T2969701,Cancelled by Customer,C1401223,Auto,Delhi Gate,Chandni Chowk,1.0,Change of plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-10,12:56:23,T6336375,Cancelled by Customer,C7874381,Auto,Laxmi Nagar,IGNOU Road,1.0,Change of plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-07-30,10:39:07,T1997739,Cancelled by Customer,C3565599,Auto,Seelampur,Kalkaji,1.0,Change of plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-06-15,08:37:03,T9352835,Cancelled by Customer,C4457487,Auto,Manesar,Old Gurgaon,1.0,Change of plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 19 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Date                               150000 non-null  datetime64[ns]
 1   Time                               150000 non-null  object        
 2   Booking ID                         150000 non-null  object        
 3   Booking Status                     150000 non-null  object        
 4   Customer ID                        150000 non-null  object        
 5   Vehicle Type                       150000 non-null  object        
 6   Pickup Location                    150000 non-null  object        
 7   Drop Location                      150000 non-null  object        
 8   Cancelled Rides by Customer        10500 non-null   float64       
 9   Reason for cancelling by Customer  10500 non-null   object        
 10  Cancelled Rides by D

In [5]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,150000,2025-06-30 18:16:47.424000,2025-01-01 00:00:00,2025-03-31 00:00:00,2025-07-01 00:00:00,2025-09-30 00:00:00,2025-12-30 00:00:00,NaN
Cancelled Rides by Customer,10500.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
Cancelled Rides by Driver,27000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
Incomplete Rides,9000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
Booking Value,102000.0,508.295912,50.0,234.0,414.0,689.0,4277.0,395.805774
Ride Distance,102000.0,24.637012,1.0,12.46,23.72,36.82,50.0,14.002138
Driver Ratings,93000.0,4.230992,3.0,4.1,4.3,4.6,5.0,0.436871
Customer Rating,93000.0,4.404584,3.0,4.2,4.5,4.8,5.0,0.437819


In [6]:
df.columns

Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location',
       'Cancelled Rides by Customer', 'Reason for cancelling by Customer',
       'Cancelled Rides by Driver', 'Driver Cancellation Reason',
       'Incomplete Rides', 'Incomplete Rides Reason', 'Booking Value',
       'Ride Distance', 'Driver Ratings', 'Customer Rating', 'Payment Method'],
      dtype='object')

In [7]:
df.columns=df.columns.str.lower().str.replace(" ","_")

In [8]:
df.columns

Index(['date', 'time', 'booking_id', 'booking_status', 'customer_id',
       'vehicle_type', 'pickup_location', 'drop_location',
       'cancelled_rides_by_customer', 'reason_for_cancelling_by_customer',
       'cancelled_rides_by_driver', 'driver_cancellation_reason',
       'incomplete_rides', 'incomplete_rides_reason', 'booking_value',
       'ride_distance', 'driver_ratings', 'customer_rating', 'payment_method'],
      dtype='object')

In [11]:
print(df.isna().sum())
print("-------------------------")
print("Dublicates : ",df.duplicated().sum())
print("-------------------------")
print("Duplicate booking_id :", df["booking_id"].duplicated().sum())
print("-------------------------")
print(df.duplicated(subset=["customer_id", "date", "time"]).sum()) 
# 1233 rows share a booking_id with another row. They have different customers, so they are different rides, not duplicates. I kept them all.

date                                      0
time                                      0
booking_id                                0
booking_status                            0
customer_id                               0
vehicle_type                              0
pickup_location                           0
drop_location                             0
cancelled_rides_by_customer          139500
reason_for_cancelling_by_customer    139500
cancelled_rides_by_driver            123000
driver_cancellation_reason           123000
incomplete_rides                     141000
incomplete_rides_reason              141000
booking_value                         48000
ride_distance                         48000
driver_ratings                        57000
customer_rating                       57000
payment_method                        48000
dtype: int64
-------------------------
Dublicates :  0
-------------------------
Duplicate booking_id : 1233
-------------------------
0


In [12]:
df["time"] = df["time"].astype(str)

In [13]:
flag_cols = ["cancelled_rides_by_customer", "cancelled_rides_by_driver", "incomplete_rides"]
df[flag_cols] = df[flag_cols].fillna(0).astype(int)

In [15]:
df["datetime"] = pd.to_datetime(df["date"].dt.strftime("%Y-%m-%d") + " " + df["time"])
df["hour"]     = df["datetime"].dt.hour
df["weekday"]  = df["datetime"].dt.day_name()
df["month"]    = df["datetime"].dt.to_period("M").astype(str)

In [17]:
df.insert(0, "ride_id", range(1, len(df) + 1))
df["is_dup_booking_id"] = df["booking_id"].duplicated(keep=False)

In [19]:
df["fare_per_km"] = (df["booking_value"] / df["ride_distance"]).round(2)
q1, q3 = df["booking_value"].quantile([0.25, 0.75])
df["is_high_fare"] = df["booking_value"] > q3 + 1.5 * (q3 - q1)

In [23]:
completed = df[df["booking_status"] == "Completed"]

In [24]:
def show(title, obj):
    print("\n" + "=" * 70 + "\n" + title + "\n" + "=" * 70)
    print(obj) 

# Questions

In [25]:
# Q1. How many rows, unique booking IDs and unique customers are in the dataset?
q1_res = pd.Series({"total_rows": len(df),
                    "unique_booking_ids": df["booking_id"].nunique(),
                    "unique_customers": df["customer_id"].nunique()})
show("Q1", q1_res)


Q1
total_rows            150000
unique_booking_ids    148767
unique_customers      104114
dtype: int64


In [27]:
# Q2. What is the distribution of bookings by status (count and %)?
q2 = pd.DataFrame({"rides": df["booking_status"].value_counts(),
                   "pct": df["booking_status"].value_counts(normalize=True).mul(100).round(2)})
show("Q2", q2)


Q2
                       rides   pct
booking_status                    
Completed              93000  62.0
Cancelled by Driver    27000  18.0
Cancelled by Customer  10500   7.0
No Driver Found        10500   7.0
Incomplete              9000   6.0


In [28]:
# Q3. What are the completion, cancellation, no-driver-found and incomplete rates?
rates = df["booking_status"].value_counts(normalize=True).mul(100).round(2)
q3 = pd.Series({"completion_pct": rates["Completed"],
                "cancellation_pct": rates["Cancelled by Customer"] + rates["Cancelled by Driver"],
                "no_driver_pct": rates["No Driver Found"],
                "incomplete_pct": rates["Incomplete"]})
show("Q3", q3)


Q3
completion_pct      62.0
cancellation_pct    25.0
no_driver_pct        7.0
incomplete_pct       6.0
dtype: float64


In [29]:
# Q4. What are the overall KPIs: total revenue, average fare, average distance and revenue per km?
q4 = pd.Series({"total_revenue": completed["booking_value"].sum(),
                "avg_fare": round(completed["booking_value"].mean(), 2),
                "avg_distance_km": round(completed["ride_distance"].mean(), 2),
                "revenue_per_km": round(completed["booking_value"].sum() / completed["ride_distance"].sum(), 2)})
show("Q4", q4)



Q4
total_revenue      47260574.00
avg_fare                508.18
avg_distance_km          26.00
revenue_per_km           19.54
dtype: float64


In [30]:
# Q5. How do rides, revenue, average fare and revenue per km compare across vehicle types?
q5 = (completed.groupby("vehicle_type")
      .agg(completed_rides=("ride_id", "size"),
           revenue=("booking_value", "sum"),
           avg_fare=("booking_value", "mean"),
           total_km=("ride_distance", "sum"))
      .assign(revenue_per_km=lambda x: x["revenue"] / x["total_km"],
              revenue_share_pct=lambda x: x["revenue"] / x["revenue"].sum() * 100)
      .drop(columns="total_km")
      .round(2)
      .sort_values("revenue", ascending=False))
show("Q5", q5)


Q5
               completed_rides     revenue  avg_fare  revenue_per_km  \
vehicle_type                                                           
Auto                     23155  11727615.0    506.48           19.49   
Go Mini                  18549   9411418.0    507.38           19.52   
Go Sedan                 16676   8538560.0    512.03           19.71   
Bike                     14034   7144913.0    509.11           19.58   
Premier Sedan            11252   5733655.0    509.57           19.64   
eBike                     6551   3298157.0    503.46           19.11   
Uber XL                   2783   1406256.0    505.30           19.64   

               revenue_share_pct  
vehicle_type                      
Auto                       24.81  
Go Mini                    19.91  
Go Sedan                   18.07  
Bike                       15.12  
Premier Sedan              12.13  
eBike                       6.98  
Uber XL                     2.98  


In [31]:
# Q6. What is the share of each payment method (rides and revenue)?
q6 = (completed.groupby("payment_method")
      .agg(rides=("ride_id", "size"), revenue=("booking_value", "sum"))
      .assign(pct_of_rides=lambda x: (x["rides"] / x["rides"].sum() * 100).round(2))
      .sort_values("rides", ascending=False))
show("Q6", q6)


Q6
                rides     revenue  pct_of_rides
payment_method                                 
UPI             41834  21274591.0         44.98
Cash            23114  11756071.0         24.85
Uber Wallet     11206   5669070.0         12.05
Credit Card      9320   4753767.0         10.02
Debit Card       7526   3807075.0          8.09


In [32]:
# Q7. What is the monthly revenue trend and the month-over-month (MoM) % change?
q7 = completed.groupby("month").agg(rides=("ride_id", "size"), revenue=("booking_value", "sum"))
q7["prev_month_revenue"] = q7["revenue"].shift(1)
q7["mom_pct"] = q7["revenue"].pct_change().mul(100).round(2)
show("Q7", q7)


Q7
         rides    revenue  prev_month_revenue  mom_pct
month                                                 
2025-01   8189  4104741.0                 NaN      NaN
2025-02   7130  3653416.0           4104741.0   -11.00
2025-03   7954  4174900.0           3653416.0    14.27
2025-04   7632  3885255.0           4174900.0    -6.94
2025-05   7905  3935207.0           3885255.0     1.29
2025-06   7757  3964298.0           3935207.0     0.74
2025-07   7926  3961939.0           3964298.0    -0.06
2025-08   7780  3867753.0           3961939.0    -2.38
2025-09   7542  3820536.0           3867753.0    -1.22
2025-10   7905  4008403.0           3820536.0     4.92
2025-11   7659  3938458.0           4008403.0    -1.74
2025-12   7621  3945668.0           3938458.0     0.18


In [33]:
# Q8. What are the top reasons customers cancel rides?
q8 = (df.loc[df["cancelled_rides_by_customer"] == 1, "reason_for_cancelling_by_customer"]
      .value_counts().to_frame("cancellations"))
q8["pct"] = (q8["cancellations"] / q8["cancellations"].sum() * 100).round(2)
show("Q8", q8)


Q8
                                              cancellations    pct
reason_for_cancelling_by_customer                                 
Wrong Address                                          2362  22.50
Change of plans                                        2353  22.41
Driver is not moving towards pickup location           2335  22.24
Driver asked to cancel                                 2295  21.86
AC is not working                                      1155  11.00


In [34]:
q9 = (df.loc[df["cancelled_rides_by_driver"] == 1, "driver_cancellation_reason"]
      .value_counts().to_frame("cancellations"))
q9["pct"] = (q9["cancellations"] / q9["cancellations"].sum() * 100).round(2)
show("Q9", q9)


Q9
                                     cancellations    pct
driver_cancellation_reason                               
Customer related issue                        6837  25.32
The customer was coughing/sick                6751  25.00
Personal & Car related issues                 6726  24.91
More than permitted people in there           6686  24.76


In [35]:
# Q10. What are the cancellation, no-driver and incomplete rates by vehicle type?
q10 = (pd.crosstab(df["vehicle_type"], df["booking_status"], normalize="index")
       .mul(100).round(2))
q10.insert(0, "total_rides", df["vehicle_type"].value_counts())
show("Q10", q10.sort_values("total_rides", ascending=False))


Q10
booking_status  total_rides  Cancelled by Customer  Cancelled by Driver  \
vehicle_type                                                              
Auto                  37419                   7.16                17.75   
Go Mini               29806                   7.04                17.88   
Go Sedan              27141                   6.75                18.54   
Bike                  22517                   6.99                18.11   
Premier Sedan         18111                   6.99                17.94   
eBike                 10557                   6.85                18.06   
Uber XL                4449                   7.35                17.13   

booking_status  Completed  Incomplete  No Driver Found  
vehicle_type                                            
Auto                61.88        6.04             7.16  
Go Mini             62.23        6.09             6.76  
Go Sedan            61.44        6.05             7.22  
Bike                62.33        5

In [36]:
# Q11. What are the peak demand hours (rides, revenue and rank by hour)?
q11 = df.groupby("hour").agg(rides=("ride_id", "size"))
q11["revenue"] = completed.groupby("hour")["booking_value"].sum()
q11["demand_rank"] = q11["rides"].rank(method="min", ascending=False).astype(int)
show("Q11", q11)


Q11
      rides    revenue  demand_rank
hour                               
0      1373   435728.0           21
1      1360   425933.0           22
2      1339   434751.0           23
3      1383   433713.0           20
4      1321   418249.0           24
5      2786   856321.0           18
6      4160  1344912.0           17
7      5450  1675816.0           15
8      6861  2183749.0           13
9      8234  2631271.0            8
10     9577  3057782.0            6
11     8390  2637838.0            7
12     7006  2231782.0           12
13     5470  1684453.0           14
14     7031  2156802.0           11
15     8202  2564322.0            9
16     9633  3012213.0            4
17    11044  3522394.0            3
18    12397  3908167.0            1
19    11047  3395818.0            2
20     9630  3035369.0            5
21     8103  2625391.0           10
22     5441  1693633.0           16
23     2762   894167.0           19


In [37]:
# Q12. How do rides and revenue vary by day of the week?
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
q12 = df.groupby("weekday").agg(rides=("ride_id", "size")).reindex(days)
q12["revenue"] = completed.groupby("weekday")["booking_value"].sum()
show("Q12", q12)


Q12
           rides    revenue
weekday                    
Monday     21274  8699700.0
Tuesday    21746  6282103.0
Wednesday  21337  5866192.0
Thursday   21526  5994406.0
Friday     21100  5980247.0
Saturday   21384  5994234.0
Sunday     21633  8443692.0


In [38]:
# Q13. Which hours have the highest "No Driver Found" rate (supply shortage)?
q13 = (df.assign(no_driver=df["booking_status"].eq("No Driver Found"))
       .groupby("hour")
       .agg(rides=("ride_id", "size"), no_driver=("no_driver", "sum"), no_driver_pct=("no_driver", "mean")))
q13["no_driver_pct"] = (q13["no_driver_pct"] * 100).round(2)
show("Q13", q13.sort_values("no_driver_pct", ascending=False).head(5))


Q13
      rides  no_driver  no_driver_pct
hour                                 
1      1360        110           8.09
2      1339        107           7.99
12     7006        529           7.55
6      4160        308           7.40
16     9633        696           7.23


In [39]:
# Q14. What are the top 10 pickup locations by rides and revenue?
q14 = df.groupby("pickup_location").agg(rides=("ride_id", "size"))
q14["revenue"] = completed.groupby("pickup_location")["booking_value"].sum()
show("Q14", q14.sort_values("rides", ascending=False).head(10))


Q14
                  rides   revenue
pickup_location                  
Khandsa             949  309041.0
Barakhamba Road     946  309713.0
Saket               931  255286.0
Badarpur            921  298309.0
Pragati Maidan      920  270546.0
Madipur             919  283263.0
AIIMS               918  296028.0
Mehrauli            915  276736.0
Dwarka Sector 21    914  287858.0
Pataudi Chowk       907  306774.0


In [41]:
# Q15. What are the top 10 most popular routes (pickup to drop)?
q15 = (df.assign(route=df["pickup_location"] + " -> " + df["drop_location"])
       .groupby("route").size().sort_values(ascending=False).head(10).to_frame("rides"))
show("Q15", q15)


Q15
                                  rides
route                                  
DLF City Court -> Bhiwadi            17
Janakpuri -> Faridabad Sector 15     16
Akshardham -> RK Puram               16
Rithala -> Udyog Vihar Phase 4       15
Ghaziabad -> Badshahpur              15
Vatika Chowk -> Rithala              15
Jor Bagh -> Rohini East              15
South Extension -> Gwal Pahari       14
Mayur Vihar -> Vidhan Sabha          14
Connaught Place -> Paharganj         14


In [42]:
# Q16. Which pickup locations have the highest cancellation rate (min. 100 rides)?
q16 = (df.assign(cancelled=df["booking_status"].isin(["Cancelled by Customer", "Cancelled by Driver"]))
       .groupby("pickup_location")
       .agg(rides=("ride_id", "size"), cancel_pct=("cancelled", "mean")))
q16["cancel_pct"] = (q16["cancel_pct"] * 100).round(2)
show("Q16", q16[q16["rides"] >= 100].sort_values("cancel_pct", ascending=False).head(10))


Q16
                     rides  cancel_pct
pickup_location                       
Vinobapuri             823       29.65
Akshardham             839       29.56
Munirka                815       28.47
Qutub Minar            823       28.31
Kadarpur               844       28.20
Faridabad Sector 15    831       28.04
GTB Nagar              872       27.75
Shahdara               820       27.68
Nehru Place            885       27.57
Karkarduma             836       27.51


In [43]:
# Q17. Who are the top 10 customers by total spending?
q17 = (completed.groupby("customer_id")
       .agg(completed_rides=("ride_id", "size"),
            total_spent=("booking_value", "sum"),
            avg_fare=("booking_value", "mean"))
       .round(1).sort_values("total_spent", ascending=False).head(10))
show("Q17", q17)


Q17
             completed_rides  total_spent  avg_fare
customer_id                                        
C7828101                   6       7683.0    1280.5
C7186567                   3       6101.0    2033.7
C2536937                   3       6019.0    2006.3
C9494011                   3       5966.0    1988.7
C3446144                   3       5589.0    1863.0
C9610969                   3       5494.0    1831.3
C5235759                   3       5379.0    1793.0
C1578062                   3       5368.0    1789.3
C3780888                   3       5323.0    1774.3
C3791237                   3       5292.0    1764.0


In [44]:
# Q18. How are customers segmented by number of completed rides (1, 2, 3+), and what is each segment's revenue share?
cust = completed.groupby("customer_id").agg(rides=("ride_id", "size"), spent=("booking_value", "sum"))
cust["segment"] = pd.cut(cust["rides"], bins=[0, 1, 2, float("inf")], labels=["1 ride", "2 rides", "3+ rides"])
q18 = (cust.groupby("segment", observed=True)
       .agg(customers=("rides", "size"), revenue=("spent", "sum")))
q18["pct_customers"] = (q18["customers"] / q18["customers"].sum() * 100).round(2)
q18["pct_revenue"] = (q18["revenue"] / q18["revenue"].sum() * 100).round(2)
show("Q18", q18)


Q18
          customers     revenue  pct_customers  pct_revenue
segment                                                    
1 ride        47581  24246832.0          73.68        51.30
2 rides        5650   5691207.0           8.75        12.04
3+ rides      11344  17322535.0          17.57        36.65


In [45]:
# Q19. What are the average driver and customer ratings by vehicle type?
q19 = (completed.groupby("vehicle_type")
       .agg(rated_rides=("ride_id", "size"),
            avg_driver_rating=("driver_ratings", "mean"),
            avg_customer_rating=("customer_rating", "mean"),
            driver_4_5_plus_pct=("driver_ratings", lambda s: (s >= 4.5).mean() * 100))
       .round(2).sort_values("avg_driver_rating", ascending=False))
show("Q19", q19)


Q19
               rated_rides  avg_driver_rating  avg_customer_rating  \
vehicle_type                                                         
Uber XL               2783               4.24                 4.40   
Auto                 23155               4.23                 4.40   
Bike                 14034               4.23                 4.40   
Go Mini              18549               4.23                 4.40   
Go Sedan             16676               4.23                 4.41   
Premier Sedan        11252               4.23                 4.40   
eBike                 6551               4.23                 4.40   

               driver_4_5_plus_pct  
vehicle_type                        
Uber XL                      29.68  
Auto                         30.42  
Bike                         30.18  
Go Mini                      29.78  
Go Sedan                     30.25  
Premier Sedan                30.83  
eBike                        29.54  


In [46]:
# Q20. Which vehicle type ranks first in revenue each month?
m = completed.groupby(["month", "vehicle_type"])["booking_value"].sum().reset_index()
q20 = m.loc[m.groupby("month")["booking_value"].idxmax()].set_index("month")
show("Q20", q20)


Q20
        vehicle_type  booking_value
month                              
2025-01         Auto       983337.0
2025-02         Auto       892140.0
2025-03         Auto      1046352.0
2025-04         Auto      1002564.0
2025-05         Auto      1009344.0
2025-06         Auto       958368.0
2025-07         Auto       949784.0
2025-08         Auto       974995.0
2025-09         Auto       979852.0
2025-10         Auto       981039.0
2025-11         Auto       958697.0
2025-12         Auto       991143.0


In [47]:
# Q21. How many bookings, completed rides and cancellations (by customer / by driver) does each customer have?
q21 = pd.crosstab(df["customer_id"], df["booking_status"])
q21["total_bookings"] = q21.sum(axis=1)
show("Q21", q21.sort_values("total_bookings", ascending=False).head(20))


Q21
booking_status  Cancelled by Customer  Cancelled by Driver  Completed  \
customer_id                                                             
C8465201                           10                    3          4   
C9584141                           10                    3          1   
C8209134                           10                    3          1   
C7473664                           10                    3          1   
C7083877                           10                    3          1   
C2724668                           10                    3          1   
C3410789                           10                    3          1   
C4012934                           10                    3          1   
C9923284                           10                    3          1   
C5383241                           11                    2          1   
C5064948                           10                    3          1   
C3400219                           10         

In [48]:
# Q22. Which 10 customers cancel the most rides themselves, and how many rides did they complete?
q22 = q21[["total_bookings", "Completed", "Cancelled by Customer"]]
show("Q22", q22.sort_values(["Cancelled by Customer", "total_bookings"], ascending=False).head(10))


Q22
booking_status  total_bookings  Completed  Cancelled by Customer
customer_id                                                     
C1825185                    14          1                     11
C4776833                    14          1                     11
C5383241                    14          1                     11
C5458644                    14          1                     11
C7178702                    14          1                     11
C7566685                    14          1                     11
C9096622                    14          1                     11
C8465201                    17          4                     10
C1029173                    14          1                     10
C1037685                    14          1                     10


In [49]:
# Q23. How many customers have cancelled at least once, and how many cancel more often than they complete rides?
q23 = pd.Series({
    "total_customers": len(q21),
    "customers_who_cancelled": int((q21["Cancelled by Customer"] > 0).sum()),
    "pct_who_cancelled": round((q21["Cancelled by Customer"] > 0).mean() * 100, 2),
    "cancelled_more_than_completed": int((q21["Cancelled by Customer"] > q21["Completed"]).sum())})
show("Q23", q23)


Q23
total_customers                  104114.0
customers_who_cancelled            2182.0
pct_who_cancelled                     2.1
cancelled_more_than_completed      2027.0
dtype: float64


In [50]:
# Q24. What is the share (%) of each booking status for every customer (customers with 5+ bookings)?
q24 = (pd.crosstab(df["customer_id"], df["booking_status"], normalize="index").mul(100).round(2))
q24.insert(0, "total_bookings", q21["total_bookings"])
show("Q24", q24[q24["total_bookings"] >= 5].sort_values("total_bookings", ascending=False).head(20))


Q24
booking_status  total_bookings  Cancelled by Customer  Cancelled by Driver  \
customer_id                                                                  
C8465201                    17                  58.82                17.65   
C4776833                    14                  78.57                14.29   
C4554642                    14                  71.43                21.43   
C7416426                    14                  71.43                21.43   
C4789573                    14                  71.43                21.43   
C4788468                    14                  71.43                21.43   
C2142419                    14                  71.43                21.43   
C7449524                    14                  71.43                21.43   
C4762730                    14                  71.43                21.43   
C7473664                    14                  71.43                21.43   
C2228359                    14                  71.43      